# 1. Data tables

Every table the monitoring layer reads, as a DataFrame. 

**Derived** (`dashboard/models.py`): `MetricsDaily`, `MetricsParticipant`,
`MetricsCohort`, `Alert` - what `/api/monitor/*` serves. They hold no collected
data and are rebuilt by `manage.py recompute_metrics`.

**Raw** (`backend/app/models.py`): the ten `app_*` tables the `dashboard/data/*`
modules actually import. `StressSample` and `EventDay` are excluded - nothing
under `dashboard/` reads either.

Read-only throughout. With `SYNTHETIC_DATA = True` no database is contacted.

In [1]:
# Set SYNTHETIC_DATA in monitor_common.py to swap fixture for ORM.
from monitor_common import *

print("data source:", describe_source())

data source: fixture fixture_cohort.json - 14 participants, seed 17, anchored 2026-09-16


In [2]:
# Every protocol constant comes from dashboard/data/config.py, the single
# authority. Nothing in these notebooks redefines one as a literal.
pd.DataFrame(
    [{"constant": "STUDY_DAYS", "value": STUDY_DAYS},
     {"constant": "RUN_IN_DAYS", "value": RUN_IN_DAYS},
     {"constant": "PARTICIPANT_TZ", "value": str(PARTICIPANT_TZ)},
     {"constant": "WAKING_WINDOW (hours)", "value": f"{WAKING_WINDOW_START_HOUR}-{WAKING_WINDOW_END_HOUR}"},
     {"constant": "JITAI_COOLDOWN_MINUTES", "value": JITAI_COOLDOWN_MINUTES},
     {"constant": "DAILY_PROMPT_CAP", "value": DAILY_PROMPT_CAP},
     {"constant": "THRESHOLD_QUANTILE", "value": THRESHOLD_QUANTILE},
     {"constant": "MSSD_WINDOW", "value": MSSD_WINDOW},
     {"constant": "OUTCOME_WINDOW_HOURS", "value": OUTCOME_WINDOW_HOURS},
     {"constant": "RATE_MIN_PARTICIPANTS", "value": RATE_MIN_PARTICIPANTS},
     {"constant": "RATE_MIN_UNITS", "value": RATE_MIN_UNITS},
     {"constant": "BENCHMARKS", "value": str(BENCHMARKS)}])

,constant,value
0,STUDY_DAYS,35
1,RUN_IN_DAYS,7
2,PARTICIPANT_TZ,America/New_York
3,WAKING_WINDOW (hours),8-22
4,JITAI_COOLDOWN_MINUTES,60
5,DAILY_PROMPT_CAP,4
6,THRESHOLD_QUANTILE,0.8
7,MSSD_WINDOW,3
8,OUTCOME_WINDOW_HOURS,2
9,RATE_MIN_PARTICIPANTS,10


## Derived metric tables (`dashboard_*`)

In [3]:
# One row per participant per study day - the densest table, and what the
# Stage 2 heatmap is built from. A null here is STRUCTURAL, not zero: a day
# outside the participant's window carries is_active_day=False with every
# metric NULL, while an active day with no activity carries 0. Never fillna(0).
metrics_daily_df.head()

,id,user_id,study_day,local_date,is_run_in,is_active_day,computed_at,item_bank_version,ema_scheduled_n,ema_jitai_n,...,prompt_dismissed_n,outcome_captured_n,wear_valid_pct,wear_gap_pct,gaps_gt2h_n,max_gap_min,hr_minutes_valid,last_sync_age_h_eod,clock_skew_p95_ms,delivery_failures_n
0,1,1001,0,2026-08-13,True,True,2026-09-16 10:00:00+00:00,v1,3,0,...,NaN,0,NaN,NaN,NaN,NaN,0,NaN,NaN,0
1,2,1001,1,2026-08-14,True,True,2026-09-16 10:00:00+00:00,v1,2,0,...,NaN,0,NaN,NaN,NaN,NaN,0,NaN,NaN,0
2,3,1001,2,2026-08-15,True,True,2026-09-16 10:00:00+00:00,v1,2,0,...,NaN,0,NaN,NaN,NaN,NaN,0,NaN,NaN,0
3,4,1001,3,2026-08-16,True,True,2026-09-16 10:00:00+00:00,v1,2,0,...,NaN,0,NaN,NaN,NaN,NaN,0,NaN,NaN,0
4,5,1001,4,2026-08-17,True,True,2026-09-16 10:00:00+00:00,v1,4,0,...,NaN,0,NaN,NaN,NaN,NaN,0,NaN,NaN,0


In [4]:
# One row per participant. This is what orders the Stage 2 call list.
# risk_score is null, not zero, for anyone the study is not currently asking
# anything of (pre-enrolment, complete, withdrawn) - otherwise every term reads
# as maximally bad for them and they dominate the top of the list.
metrics_participant_df.head()

,id,user_id,computed_at,enrolled_at,day1_date,study_day_now,phase,is_enrolled_snapshot,first_seen_not_enrolled_at,last_ema_at,...,risk_components,slot_coverage_rate,slot_coverage_num,slot_coverage_den,prompt_response_rate,prompt_response_num,prompt_response_den,wear_rate,wear_num,wear_den
0,1,1001,2026-09-16 10:00:00+00:00,2026-08-13 14:00:00+00:00,2026-08-13,34,complete,True,NaT,2026-09-15 21:00:00+00:00,...,NaN,NaN,0,0,NaN,0,0,NaN,0,0
1,2,1002,2026-09-16 10:00:00+00:00,2026-08-26 14:00:00+00:00,2026-08-26,21,withdrawn,False,2026-09-10 10:00:00+00:00,2026-09-15 21:00:00+00:00,...,NaN,NaN,0,0,NaN,0,0,NaN,0,0
2,3,1003,2026-09-16 10:00:00+00:00,2026-08-29 14:00:00+00:00,2026-08-29,18,mrt,True,NaT,2026-09-15 21:00:00+00:00,...,NaN,NaN,0,0,NaN,0,0,NaN,0,0
3,4,1004,2026-09-16 10:00:00+00:00,2026-09-10 14:00:00+00:00,2026-09-10,6,run_in,True,NaT,2026-09-15 21:00:00+00:00,...,NaN,NaN,0,0,NaN,0,0,NaN,0,0
4,5,1005,2026-09-16 10:00:00+00:00,2026-09-10 14:00:00+00:00,2026-09-10,6,run_in,True,NaT,2026-09-15 21:00:00+00:00,...,NaN,NaN,0,0,NaN,0,0,NaN,0,0


In [5]:
# Append-only, one row per compute run per phase filter. cohort_latest is the
# row GET /api/monitor/cohort?phase= returns. benchmarks / series_14d /
# integrity / funnel stay JSON documents because the row is always read whole.
cohort_latest[["as_of", "phase_filter", "n_participants", "n_active"]]

,as_of,phase_filter,n_participants,n_active
0,2026-09-16 10:00:00+00:00,all,14,11
1,2026-09-16 10:00:00+00:00,phase1,0,0
2,2026-09-16 10:00:00+00:00,phase2,14,11


In [6]:
# Open and resolved alerts. A null user_id is a COHORT-scoped alert, not
# missing data: a fact about the engine or the pipeline fires once for the
# cohort rather than once per participant, which would bury the view.
alerts_df[["fired_at", "severity", "rule_id", "user_id", "resolved_at"]]

,fired_at,severity,rule_id,user_id,resolved_at
0,2026-09-16 10:00:00+00:00,critical,runin_violation,NaN,NaT
1,2026-09-16 10:00:00+00:00,critical,sync_stale,NaN,NaT
2,2026-09-16 10:00:00+00:00,critical,cooldown_violation,1007.0,NaT
3,2026-09-16 10:00:00+00:00,critical,cap_exceeded,1008.0,NaT
4,2026-09-16 10:00:00+00:00,high,no_ema_48h,1009.0,NaT
5,2026-09-16 10:00:00+00:00,high,wear_low,1010.0,NaT
6,2026-09-16 10:00:00+00:00,warning,slot_coverage_low,1011.0,NaT


## Raw source tables (`app_*`)

In [7]:
# Participants. email, names and password are deliberately not loaded - they
# are direct identifiers and no monitoring module reads them. The push token is
# reduced to a boolean because its presence is the only part the pipeline cares
# about: a missing token is why a prompt silently fails to deliver.
users_df

,user_id,is_enrolled,enrolled_at,gender,birthdate
0,1001,True,2026-08-13 14:00:00+00:00,female,2008-06-24
1,1002,False,2026-08-26 14:00:00+00:00,female,2007-07-11
2,1003,True,2026-08-29 14:00:00+00:00,female,2005-07-15
3,1004,True,2026-09-10 14:00:00+00:00,female,2007-03-10
4,1005,True,2026-09-10 14:00:00+00:00,other,2007-02-18
5,1006,True,2026-09-11 14:00:00+00:00,other,2008-01-21
6,1007,True,2026-09-01 14:00:00+00:00,female,2005-03-19
7,1008,True,2026-08-25 14:00:00+00:00,male,2006-08-18
8,1009,True,2026-08-23 14:00:00+00:00,male,2006-09-13
9,1010,True,2026-08-19 14:00:00+00:00,other,2008-05-12


In [8]:
# One device per participant. WearableSync is the append-only log of the sync
# clock advancing; it exists because WearableDevice.last_synced_at is a single
# mutable column, so without it a sync outage cannot be told apart from genuine
# non-wear. In production nothing writes it yet - the fixture does.
print("devices:", len(wearable_devices_df), "| sync events:", len(wearable_sync_df))
wearable_devices_df.head()

devices: 14 | sync events: 562


,id,user_id,labfront_participant_id,is_active,last_synced_at
0,1,1001,SYN-000,True,2026-09-15 14:00:00+00:00
1,2,1002,SYN-001,False,2026-09-15 11:00:00+00:00
2,3,1003,SYN-002,False,2026-09-15 01:00:00+00:00
3,4,1004,SYN-003,True,2026-09-15 11:00:00+00:00
4,5,1005,SYN-004,True,2026-09-16 05:00:00+00:00


In [9]:
# Every check-in. served_sub_item_ids records what actually reached the screen,
# which is what makes item completeness measurable at all. No free-text field is
# stored anywhere in this table - that is an IRB constraint, not an omission.
print(ema_df.groupby(["ema_type", "status"]).size().to_string())
ema_df.head()

ema_type            status   
post_prompt         completed     43
prompt_feedback     completed     43
scheduled_check_in  completed    751


,id,user_id,prompt_id,ema_type,status,sent_at,responded_at,expires_at,outcome_window_start,outcome_window_end,source_jitai_log_id,served_sub_item_ids,mood,stress,energy
0,1,1001,scheduled_check_in_1,scheduled_check_in,completed,2026-08-13 13:30:00+00:00,2026-08-13 13:32:00+00:00,2026-08-13 14:00:00+00:00,NaT,NaT,NaN,"[B1_valence, B1_arousal, B1_fluctuation, B1_ch...",1.0,3.0,3.0
1,2,1001,scheduled_check_in_2,scheduled_check_in,completed,2026-08-13 15:31:00+00:00,2026-08-13 15:37:00+00:00,2026-08-13 16:01:00+00:00,NaT,NaT,NaN,"[B1_valence, B1_arousal, B1_fluctuation, B1_ch...",2.0,5.0,1.0
2,3,1001,scheduled_check_in_3,scheduled_check_in,completed,2026-08-13 17:22:00+00:00,2026-08-13 17:23:00+00:00,2026-08-13 17:52:00+00:00,NaT,NaT,NaN,"[B1_valence, B1_arousal, B1_fluctuation, B1_ch...",2.0,2.0,4.0
3,4,1001,scheduled_check_in_4,scheduled_check_in,completed,2026-08-14 21:56:00+00:00,2026-08-14 22:06:00+00:00,2026-08-14 22:26:00+00:00,NaT,NaT,NaN,"[B1_valence, B1_arousal, B1_fluctuation, B1_ch...",4.0,4.0,5.0
4,5,1001,scheduled_check_in_5,scheduled_check_in,completed,2026-08-14 23:04:00+00:00,2026-08-14 23:13:00+00:00,2026-08-14 23:34:00+00:00,NaT,NaT,NaN,"[B1_valence, B1_arousal, B1_fluctuation, B1_ch...",1.0,5.0,5.0


In [10]:
# One row per answered sub-item. Joined to the frozen item bank (v1), this is
# what Stage 3's completeness matrix scores: answered over askable.
ema_item_responses_df.head()

,id,ema_id,item_id,sub_item_id,response_type,value_numeric,value_choice,value_choices
0,1,1,B1,B1_valence,likert,1.0,NaN,NaN
1,2,1,B1,B1_arousal,likert,3.0,NaN,NaN
2,3,1,B1,B1_fluctuation,likert,3.0,NaN,NaN
3,4,1,B1,B1_change,single_choice,NaN,Somewhat better,NaN
4,5,1,B1,B1_affect_anxious,likert,2.0,NaN,NaN


In [11]:
# The reminder log. It is what lets a missed slot be classified: covered,
# reminded-but-skipped, or silent. Only the third is a scheduler failure, and
# nobody should be phoned about it.
checkin_reminders_df.head()

,id,user_id,sent_at,daily_count_at_send
0,1,1001,2026-08-13 19:30:00+00:00,3
1,2,1001,2026-08-13 21:30:00+00:00,4
2,3,1001,2026-08-13 23:30:00+00:00,5
3,4,1001,2026-08-14 13:30:00+00:00,0
4,5,1001,2026-08-14 15:30:00+00:00,1


In [12]:
# Every decision point, sent or not - send_prompt is the gate, so the full
# table is the denominator and the send_prompt=True subset is the dose.
# threshold_at_decision records what observed_mssd was compared against;
# threshold_source is 'engine' when the live decision wrote it.
print("decision points:", len(jitai_log_df), "| sent:", int(jitai_log_df["send_prompt"].sum()))
print(jitai_log_df["threshold_source"].value_counts(dropna=False).to_string())
jitai_log_df.head()

decision points: 428 | sent: 73
threshold_source
engine           288
reconstructed    140


,id,user_id,prompt_id,triggered_at,trigger_reason,trigger_signal,decision_point_id,ema_id,observed_mssd,threshold_at_decision,...,receipt_event_id,delivery_status,delivery_error,receipt_platform,receipt_app_state,eligible_prompt_ids,evaluated_items,matched_categories,category_drawn,fallback_reason
0,1,1001,tmpl_01,2026-08-18 13:04:00+00:00,prompt sent,mssd,dp-1001-2026-08-18-0,NaN,0.539,2.532,...,rcpt-1,received_on_device,NaN,android,foreground,"[tmpl_01, tmpl_02, tmpl_03]","{'B1_valence': 4, 'B2_stress': 7}",[stress],stress,NaN
1,2,1001,tmpl_06,2026-08-20 13:40:00+00:00,below within-person threshold,mssd,dp-1001-2026-08-20-0,NaN,1.679,3.221,...,NaN,not_sent,NaN,NaN,NaN,"[tmpl_01, tmpl_02, tmpl_03]","{'B1_valence': 1, 'B2_stress': 5}",[],NaN,NaN
2,3,1001,tmpl_07,2026-08-20 16:19:00+00:00,below within-person threshold,mssd,dp-1001-2026-08-20-1,NaN,0.931,2.966,...,NaN,not_sent,NaN,NaN,NaN,"[tmpl_01, tmpl_02, tmpl_03]","{'B1_valence': 2, 'B2_stress': 5}",[],NaN,NaN
3,4,1001,tmpl_08,2026-08-20 19:28:00+00:00,below within-person threshold,mssd,dp-1001-2026-08-20-2,NaN,2.800,2.855,...,NaN,not_sent,NaN,NaN,NaN,"[tmpl_01, tmpl_02, tmpl_03]","{'B1_valence': 7, 'B2_stress': 6}",[],NaN,NaN
4,5,1001,tmpl_04,2026-08-20 22:24:00+00:00,below within-person threshold,mssd,dp-1001-2026-08-20-3,NaN,0.662,3.046,...,NaN,not_sent,NaN,NaN,NaN,"[tmpl_01, tmpl_02, tmpl_03]","{'B1_valence': 7, 'B2_stress': 2}",[],NaN,NaN


In [13]:
# Engagement backs the opened / acted / dismissed counts. PhoneTelemetry
# carries occurred_at (device clock) beside recorded_at (server clock); their
# difference is clock skew, which is SIGNED - a device running ahead of the
# server is a real diagnostic condition, not an error.
print(engagement_log_df["event_type"].value_counts().to_string())
phone_telemetry_df.head()

event_type
notification_tapped       39
ema_opened                21
notification_dismissed    15


,id,user_id,session_id,event_type,occurred_at,recorded_at,screen_name,latency_ms,metadata
0,1,1001,s-1001-2026-08-18,compose_open,2026-08-18 13:50:00+00:00,2026-08-18 13:50:03+00:00,checkin,355,{'build': '1.4.2'}
1,2,1001,s-1001-2026-08-20,compose_open,2026-08-20 15:43:00+00:00,2026-08-20 15:43:07+00:00,checkin,110,{'build': '1.4.2'}
2,3,1001,s-1001-2026-08-20,compose_open,2026-08-20 22:15:00+00:00,2026-08-20 22:15:08+00:00,checkin,544,{'build': '1.4.2'}
3,4,1001,s-1001-2026-08-21,compose_open,2026-08-21 13:38:00+00:00,2026-08-21 13:38:04+00:00,checkin,466,{'build': '1.4.2'}
4,5,1001,s-1001-2026-08-22,compose_open,2026-08-22 15:10:00+00:00,2026-08-22 15:10:05+00:00,checkin,360,{'build': '1.4.2'}


In [14]:
# The only table that can run to millions of rows - production ingests heart
# rate every 15 seconds. HR_DAYS bounds it to a trailing window; the full count
# prints beside the loaded count so the truncation is always visible.
print(f"rows in table: {hr_total} | loaded (HR_DAYS={HR_DAYS}): {len(heart_rate_df)}")
heart_rate_df.head()

rows in table: 127469 | loaded (HR_DAYS=14): 127469


,id,user_id,timestamp,bpm,source
0,1,1001,2026-09-03 12:00:00+00:00,74,garmin_labfront
1,2,1001,2026-09-03 12:01:00+00:00,69,garmin_labfront
2,3,1001,2026-09-03 12:02:00+00:00,80,garmin_labfront
3,4,1001,2026-09-03 12:03:00+00:00,73,garmin_labfront
4,5,1001,2026-09-03 12:04:00+00:00,72,garmin_labfront


## Summary

In [15]:
# Every frame loaded, with its shape. An all-zero derived block means
# recompute_metrics has never run against this database.
table_summary()

,table,rows,columns
0,dashboard_metricsdaily,256,39
1,dashboard_metricsparticipant,14,23
2,dashboard_metricscohort,3,17
3,dashboard_alert,7,7
4,app_user,14,5
5,app_wearabledevice,14,5
6,app_wearablesync,562,6
7,app_ema,837,15
8,app_emaitemresponse,15409,8
9,app_checkinreminder,600,4
